In [ ]:
#------------
#Setup 
#------------

import json
from typing import List
import langchain 
from langchain_community import document_loaders

from unstructured.partition.pdf import partition_pdf 
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage

#--------------------
#! winget install --id UB-Mannheim.TesseractOCR
import os
import pytesseract
import unstructured_pytesseract
r'D:\RAG\Tesseract\Tesseract-OCR'

tesseract_dir = r'D:\RAG\Tesseract\Tesseract-OCR'
tesseract_exe = tesseract_dir + r'\tesseract.exe'

# Add to PATH for this process (covers subprocess calls generally)
os.environ["PATH"] += os.pathsep + tesseract_dir

# Also set explicitly on both known wrapper packages
pytesseract.pytesseract.tesseract_cmd = tesseract_exe
unstructured_pytesseract.pytesseract.tesseract_cmd = tesseract_exe 

print(pytesseract.get_tesseract_version()) 
#--------------------

In [ ]:
import json
from typing import List
import langchain 
from langchain_community import document_loaders

from unstructured.partition.pdf import partition_pdf 
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage

#--------------------
#! winget install --id UB-Mannheim.TesseractOCR
import os
import pytesseract
import unstructured_pytesseract
r'D:\RAG\Tesseract\Tesseract-OCR'

tesseract_dir = r'D:\RAG\Tesseract\Tesseract-OCR'
tesseract_exe = tesseract_dir + r'\tesseract.exe'

# Add to PATH for this process (covers subprocess calls generally)
os.environ["PATH"] += os.pathsep + tesseract_dir

# Also set explicitly on both known wrapper packages
pytesseract.pytesseract.tesseract_cmd = tesseract_exe
unstructured_pytesseract.pytesseract.tesseract_cmd = tesseract_exe 

print(pytesseract.get_tesseract_version()) 

In [ ]:
def partition_document(file_path:str):

    elements = partition_pdf(
        filename= file_path, 
        strategy = 'hi_res', #use the most accurate (but slower) processing method for extraction 
        infer_table_structure= True, #keep table as structured HTML, not jumbled text 
        extract_image_block_types = ['Image'], 
        extract_image_block_to_payload= True,  
        extract_image_block_output_dir= r'D:\RAG\Data\pdfImageData', 
      # languages = ['English', 'Hindi']  
    )

    print("extracted {len(elements)} elemets")
    return elements 
   
file_path = r'D:\RAG\Data\Attention is that all you need.pdf'
elements = partition_document(file_path)

In [ ]:
chunks[4].metadata.orig_elements[3].to_dict() 

In [ ]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    
    chunks = chunk_by_title(
        elements,
        max_characters=1800,        # ≈ 450 tokens — safe margin under 512
        new_after_n_chars=1300,     # ≈ 325 tokens — soft target
        combine_text_under_n_chars=400   # ≈ 100 tokens
    )
    
    return chunks

chunks = create_chunks_by_title(elements)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

local_path = r"D:\Gemma4 2B 4B\gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(local_path)

tokenizer = processor.tokenizer

model = AutoModelForImageTextToText.from_pretrained(
    local_path,
    low_cpu_mem_usage=True,  
)

model.to('cpu')



def generate(prompt_text, tokenizer, max_new_tokens= 1024):

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id= model.tokenizer.eos_token_id 
    )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    
    return model.tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
def seperate_content_types(chunks): 
    
    """Analyze what types of content are in a chunk""" 
    
    content_data = {'text': chunks.text, 
                    'tables': [], 
                    'images': [],
                    'types': ['text']     
                    }
    
    if hasattr(chunks, 'metadata') and hasattr(chunks.metadata, 'orig_elements'): 
        
        for element in chunks.metadata.orig_elements: 
            element_type = type(element).__name__
            
            #Handel tables 
            if element_type == 'Table': 
                content_data['types'].append('table') 

                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html) 
                
            elif element_type ==  'Image': 
                
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image'): 
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64) 
         
    content_data['types'] = list(set(content_data['types']))  
    
    return content_data

In [ ]:
from typing import List

def create_ai_enhansed_summary(llm, tokenizer, text: str, tables: List[str], images: List[str]) -> str: 
    """Create-AI-enhanced summary for mixed contnt"""
    
    try: 

        prompt_text = f"""You are creating a searchable description for documet content retrival.
        
        Content to analyze: 
        Text Content: 
        {text} 
        
        """
        if tables:
            prompt_text += 'TABLE Content:\n' 
            
            for i, table in enumerate(tables): 
                prompt_text += f"Table {i+1}\n{table}\n\n" 
                
                prompt_text += """
                YOUR TASK: 
                Generate a comprehensive, searchable  description that covers: 
                
                1. The table content i.e. key facts, numbers, and data points from text and tables. 
                2. Main topics and concepts discussed 
                3. Questions this content could answer
                4. Visual content analysis (chart, diagrams, patterns in imags)
                5. Alternative search terms user might use 
                
                Make it detailed and searchable - prioritize findability over brevity.
                
                Searchable Description:"""
        
                
        message_content = [
                    {"role": 'system',
                    'content':[{'type': 'text', 'text': 'You are a enterprise ai assistant to a company, you reffer the user as Sir.'}]
                    },

                    {
                    "role": "user",
                    "content": [{"type": "text", "text": prompt_text}]
                    }
                ]

        if images:
            
            for image_base64 in images: 
                message_content[1]['content'].append({
                    'type': "image_url", 
                    'image_url': {'url': f'data:image/jpeg;base64,{image_base64}'}
                    }) 
                               
        response = llm.generate(message_content, tokenizer)
        
    except Exception as e: 
      print(f" Error: AI Summary failed: {e}")
      
      summary = f"{text[:300]}..." 
      if tables: 
         summary += f" [Content {len(tables)} table(s)]"
      if images: 
          summary += f" [Content {len(images)} image(s)]"
      
      
      return summary
   

In [ ]:
def summarise_chunks(llm, tokenizer, chunks): 
    """Proessing all chunsk with AI summaries"""

    print("Processing chunks with AI summaries...")
    print()
    
    langchain_document = [] 
    total_chunks = len(chunks) 

    for i, chunk in enumerate(chunks):
        
        current_chunk = i+1 
        print(f"  Processing chunk {current_chunk}/{total_chunks}")
    
        content_data = seperate_content_types(chunk) 
        print(f"   Types found:{content_data['types']}")
        print()
    
    if content_data['tables'] or content_data['images']: 
        
        print(f"   -> Creating AI summary for mixed content...")
        
        try: 
            enhanced_content = create_ai_enhansed_summary(llm, tokenizer,
                 content_data['text'],
                 content_data['tables'], 
                 content_data['imges']
                ) 
            
            print(f" -> AI summary created successfully")
            print(f" -> Enanced content preview: {enhanced_content[:200]}...")
            
        except Exception as e: 
            print(f"  AI summery failed {e}") 
            
            enhanced_content = content_data['text'] 
                     
    else: 
        print(f" ->Using raw text (no tables/images)") 

        enhanced_content = content_data['text'] 
            
    #Create langchain document with rich metadata      
    doc = Document(
        page_content= enhanced_content, 
        metadata = {
            'original content': json.dumps({
                'raw_text': content_data['text'], 
                'tables_html': content_data['tables'], 
                'images_base64': content_data['images']
            })
        }
    )     
       
    langchain_document.append(doc)
    print(f" Processed {len(langchain_document)} chunks") 
    return langchain_document 

processed_chunks = summarise_chunks(model, tokenizer, chunks)

In [ ]:
def export_chunks_to_json(chunks, filename='chunks_export.json'): 
    """Export processed chunks to clean JSON format"""
    
    export_data = []
    
    for i, doc in enumerate(chunks): 
        
        chunk_data = {
            'chunk_id': i+1, 
            'enhanced_content': doc.page_content,
            'metadata': {
                'original_content': json.loads(doc.metadata.get('original_content', '{}'))
            }
        }
        export_data.append(chunk_data)
        
    with open(filename, 'w', encoding = 'utf-8') as f:
        json.dump(export_data, f, indent= 2, ensure_ascii= False)      
        
    print(f'Exported {len(export_data)} chunks to {filename}')
    return export_data   
        
json_data = export_chunks_to_json(processed_chunks)

In [ ]:
import json
import re

def clean_enhanced_text(text: str) -> str:
    """Collapse repeated newlines/whitespace for embedding-ready text."""
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

with open('chunks_export.json', 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

enhanced_texts = [clean_enhanced_text(chunk['enhanced_content']) for chunk in chunks_data]

print(len(enhanced_texts))
print(enhanced_texts[0])